# K-means and clustering / Phân cụm

Tôi lưu bài này trong nhật ký học để ôn lại cách viết và đọc code.

**Nguồn / Source:** Bản Colab `Kmeans with Python.ipynb`. Các bài có tên “Bản sao” là tài liệu thực hành tham khảo; tên file không được xem là bằng chứng tác giả.

**Cách dùng / Run:** chạy các cell theo thứ tự. Thư viện được dùng: `__future__`, `google.colab`, `matplotlib.pyplot`, `numpy`, `scipy.spatial.distance`, `sklearn.cluster`.

**Execution:** ARCHIVED_SOURCE — bản mã học tập, chưa ghi nhận chạy lại đầy đủ trong đợt này. Output lịch sử được bỏ; hướng dẫn và lỗi cần xử lý nằm trong bảng kỹ thuật.

Learning archive: coursework/tutorial practice, with original code retained for study. Original tutorial references embedded in the source are retained below.

# Xây dựng đoạn code cho thuật toán Kmeans

In [ ]:
#khối mã cần thiết để nhập các thư viện triển khai & trực quan hoá thuật toán phân cụm K-means

from __future__ import print_function # Dòng này đảm bảo hoạt động print ổn định trên tất cả phiên bản Python
import numpy as np # Thêm thư viện NumPy, thường sử dụng cho toán tử, mảng, phép gán - gán là: np
import matplotlib.pyplot as plt #Nhập module pylot của thư viện maplot, cung cấp sơ đồ, trực quan hoá dữ liệu - gán là plt
from scipy.spatial.distance import cdist # nhập hàm cdist từ module spatial distance của scipy, hàm tính khoảng cách mỗi cặp trong tập nhập vào
np.random.seed(11) # cố định kq khởi tạo, nhất quán dữ liệu → Nếu không đặt seed, mỗi lần chạy K-means có thể cho kết quả khác nhau do khởi tạo tâm cụm ngẫu nhiên.


In [ ]:
# Khối mã tạo ra dữ liệu tổng hợp để chứng minh thuật toán
# K-means

means = [[2, 2], [8, 3], [3, 6]] #Xác định danh sách biểu diễn các giá trị trung bình cho 3 cụm khác nhau
cov = [[1, 0], [0, 1]] # xác định danh sách ma trận biểu diễn hiệp phương sai cho mỗi cụm, ma trận đơn vị [1, 0],[0, 1] chỉ ra rằng dữ liệu trong mỗi cụm có dạng hình cầu
N = 100 # Đặt số điểm dữ liệu cho mỗi cụm 100
X0 = np.random.multivariate_normal(means[0], cov, N) # Tạo N điểm dữ liệu cho cụm đầu tiên bằng cách sử dụng phân phối chuẩn đa biến với ma trận trung bình và hiệp phương sai đầu tiên
X1 = np.random.multivariate_normal(means[1], cov, N) # Tạo N điểm dữ liệu cho cụm thứ 2 bằng cách sử dụng phân phối chuẩn đa biến với ma trận trung bình và hiệp phương sai đầu thứ 2
X2 = np.random.multivariate_normal(means[2], cov, N) # Tạo N điểm dữ liệu cho cụm thứ 3 bằng cách sử dụng phân phối chuẩn đa biến với ma trận trung bình và hiệp phương sai thứ 3

X = np.concatenate((X0, X1, X2), axis = 0) # Kết hợp các cụm dữ liệu thành 1 mảng np X duy nhất, đối số axis xếp chồng các mảng thành chiều dọc
K = 3 # Đặt số lượng cụm dự kiến là 3

original_label = np.asarray([0]*N + [1]*N + [2]*N).T # Tạo một mảng các nhãn gốc cho mỗi điểm dữ liệu, N điểm đầu tiên gán nhãn 0, N điểm tiếp theo gán nhãn 1, N điểm cuối gán nhãn 2. T chuyển vị mặc dù mảng 1 chiều để trực quan hoá dữ liệu gốc

In [ ]:
from google.colab import drive #Nhập module ổ đĩa từ google colab, kết nối các hàm tương tác với Google drive
drive.mount('/content/drive') # gọi hàm mount từ module drive đã nhập, mount nội dung từ Google drive theo đường dẫn đã chỉ định
#Sau hàm trên có thể truy cập trực tiếp các mục trên Google drive thông qua ghi chép từ colab

In [ ]:
def kmeans_display(X, label): #Hàm này định nghĩa một hàm có tên kmeans_display, nhận hai đối số: X (các điểm dữ liệu) và label (nhãn cụm cho mỗi điểm dữ liệu).
    K = np.amax(label) + 1 #tính toán số cụm (K) bằng cách tìm giá trị lớn nhất trong mảng nhãn và cộng thêm 1. Giả định rằng các nhãn cụm được đánh chỉ mục 0.
    X0 = X[label == 0, :] #Các dòng này tạo các mảng riêng biệt cho các điểm dữ liệu thuộc về mỗi cụm (0, 1 và 2) bằng cách sử dụng chỉ mục boolean trên mảng nhãn.
    X1 = X[label == 1, :]
    X2 = X[label == 2, :]

#vẽ các điểm dữ liệu cho cụm 0, biểu diễn tọa độ x và y của các điểm trong cụm 0, 'b^' chỉ định kiểu đánh dấu: 'b' cho màu xanh lam và '^' cho các đánh dấu hình tam giác.
#markersize = 4 thiết lập kích thước của các đánh dấu.
#alpha = .8 thiết lập độ trong suốt của các đánh dấu.
    plt.plot(X0[:, 0], X0[:, 1], 'b^', markersize = 4, alpha = .8)
    plt.plot(X1[:, 0], X1[:, 1], 'go', markersize = 4, alpha = .8) #Dòng này vẽ các điểm dữ liệu cho cụm 1 bằng các vòng tròn màu xanh lá cây ('go').
    plt.plot(X2[:, 0], X2[:, 1], 'rs', markersize = 4, alpha = .8) #vẽ các điểm dữ liệu cho cụm 2 bằng các hình vuông màu đỏ ('rs').

    plt.axis('equal') #Dòng này đặt tỷ lệ của trục x và trục y bằng nhau, sao cho các hình tròn xuất hiện dưới dạng hình tròn và hình vuông xuất hiện dưới dạng hình vuông, tránh bị biến dạng.
    plt.plot() # Đây là một lệnh vẽ đồ thị rỗng, đôi khi được sử dụng để đảm bảo đồ thị được hiển thị chính xác.
    plt.show() # Lệnh này hiển thị đồ thị đã tạo.

kmeans_display(X, original_label) #Dòng này gọi hàm kmeans_display với dữ liệu kết hợp X và mảng original_label đã được tạo trước đó, trực quan hóa hiệu quả phân phối dữ liệu ban đầu với các phép gán cụm thực sự.


In [ ]:
#Mã này định nghĩa các hàm hỗ trợ cho thuật toán K-means:
#Hàm này khởi tạo các tâm cụm. Nó chọn ngẫu nhiên k điểm dữ liệu từ dữ liệu đầu vào X để làm tâm ban đầu. replace=False đảm bảo rằng mỗi điểm được chọn là duy nhất.
def kmeans_init_centers(X, k):
    return X[np.random.choice(X.shape[0], k, replace=False)]

def kmeans_assign_labels(X, centers): #Hàm này gán mỗi điểm dữ liệu cho tâm cụm gần nhất.
    D = cdist(X, centers) #tính toán khoảng cách giữa mỗi điểm dữ liệu trong X và mỗi tâm trong centers.
    return np.argmin(D, axis = 1) #tìm chỉ số khoảng cách tối thiểu cho mỗi điểm dữ liệu dọc theo trục 1 (các tâm), tương ứng với nhãn cụm đã được gán.

def kmeans_update_centers(X, labels, K): #Hàm này cập nhật các tâm cụm dựa trên các phép gán hiện tại.
    centers = np.zeros((K, X.shape[1])) #Nó khởi tạo một mảng centers với các số 0. Nó lặp qua mỗi cụm k từ 0 đến K-1.
    for k in range(K):
        Xk = X[labels == k, :] #chọn tất cả các điểm dữ liệu hiện đang được gán cho cụm k.
        centers[k,:] = np.mean(Xk, axis = 0) #tính toán giá trị trung bình của các điểm dữ liệu trong cụm k và đặt giá trị này làm tâm mới cho cụm đó.
    return centers #returns the updated array of cluster centers

#Hàm này kiểm tra xem thuật toán K-means đã hội tụ hay chưa. Nó so sánh các tâm hiện tại với các tâm mới được tính toán trong bước cập nhật. Thuật toán được coi là hội tụ nếu tập hợp các tâm hiện tại giống với tập hợp các tâm mới. Nó chuyển đổi các tâm thành các bộ và sau đó thành các tập hợp để so sánh nhằm xử lý các sai số dấu phẩy động tiềm ẩn và thứ tự của các tâm.
def has_converged(centers, new_centers):
    return (set([tuple(a) for a in centers]) == #Dòng này so sánh tập hợp các tâm cụm hiện tại với tập hợp các tâm mới được tính toán. == so sánh hai tập hợp. Nếu các tập hợp bằng nhau, điều đó có nghĩa là các tâm không thay đổi đáng kể trong lần lặp này và thuật toán được coi là đã hội tụ. Sau đó, hàm trả về True; nếu không, nó trả về False.
        set([tuple(a) for a in new_centers])) #chuyển đổi danh sách các tuple thành một tập hợp. Sử dụng một tập hợp cho phép so sánh các tâm bất kể thứ tự của chúng trong mảng.

In [ ]:
def kmeans(X, K): #định nghĩa hàm kmeanslấy dữ liệu Xvà số lượng cụm mong muốn Klàm đầu vào.
    centers = [kmeans_init_centers(X, K)] #khởi tạo một danh sách có tên là centers. Nó gọi kmeans_init_centershàm để lấy các tâm cụm ngẫu nhiên ban đầu và lưu trữ chúng làm phần tử đầu tiên trong centersdanh sách. centersDanh sách sẽ lưu trữ các tâm cụm ở mỗi lần lặp.
    labels = [] #khởi tạo một danh sách rỗng có tên là labels. Danh sách này sẽ lưu trữ các phép gán cụm cho từng điểm dữ liệu tại mỗi lần lặp.
    it = 0 #Điều này khởi tạo một bộ đếm itđể theo dõi số lần lặp lại.
    while True: #bắt đầu một vòng lặp vô hạn và sẽ tiếp tục cho đến khi thuật toán hội tụ.
        labels.append(kmeans_assign_labels(X, centers[-1])) #Bên trong vòng lặp, dòng này gọi kmeans_assign_labelshàm để gán từng điểm dữ liệu vào Xtrung tâm cụm gần nhất từ tập trung tâm cuối cùngcenters được lưu trữ trong danh sách ( centers[-1]). Các nhãn kết quả được thêm vào labelsdanh sách.
        new_centers = kmeans_update_centers(X, labels[-1], K)#Dòng này gọi kmeans_update_centershàm để tính toán các trung tâm cụm mới dựa trên các chỉ định cụm mới nhất ( labels[-1]). Các trung tâm mới được lưu trữ trong new_centersbiến.
        if has_converged(centers[-1], new_centers): Dòng này kiểm tra xem thuật toán có hội tụ hay không bằng cách so sánh tập hợp các trung tâm cuối cùng trong centersdanh sách với việc new_centerssử dụng has_convergedhàm.
            break #Nếu has_convergedhàm trả về True(có nghĩa là tâm không thay đổi đáng kể), breakcâu lệnh sẽ thoát khỏi whilevòng lặp.
        centers.append(new_centers) #Nếu thuật toán chưa hội tụ, chúng new_centerssẽ được thêm vào centersdanh sách để sử dụng trong lần lặp tiếp theo.
        it += 1 #Bộ đếm lặp itđược tăng lên.
    return (centers, labels, it) #Khi vòng lặp bị phá vỡ (đạt đến điểm hội tụ), hàm sẽ trả về một bộ chứa: Danh sách tất cả các trung tâm cụm tại mỗi lần lặp lại ( centers). Danh sách tất cả các nhiệm vụ cụm tại mỗi lần lặp lại ( labels). Tổng số lần lặp cần thiết để hội tụ ( it).


In [ ]:
(centers, labels, it) = kmeans(X, K) #gọi kmeanshàm với dữ liệu Xvà số cụm K(trước đó đã được đặt thành 3). Hàm trả về một bộ dữ liệu chứa lịch sử các tâm, lịch sử các nhãn và số lần lặp. Các dữ liệu này được giải nén vào các biến centers, labels, và it.
print('Centers found by our algorithm:') #in ra thông báo mô tả vào bảng điều khiển.
print(centers[-1]) # in ra các trung tâm cụm cuối cùng được tìm thấy bởi thuật toán K-means bạn triển khai. centers[-1]Truy cập phần tử cuối cùng trong centersdanh sách, chứa các trung tâm sau khi thuật toán hội tụ.

kmeans_display(X, labels[-1]) #Dòng này gọi kmeans_displayhàm (được định nghĩa trước đó) để trực quan hóa các điểm dữ liệu với các cụm được gán cuối cùng mà thuật toán của bạn tìm thấy. labels[-1]cung cấp nhãn cụm cho mỗi điểm dữ liệu sau khi thuật toán hội tụ.
#Về bản chất, đoạn mã này chạy triển khai K-means của bạn trên dữ liệu được tạo ra và sau đó hiển thị cho bạn các trung tâm cụm cuối cùng và sơ đồ các điểm dữ liệu được tô màu theo cụm được chỉ định của chúng.

# Thuật toán Kmeans với thư viện Sklearn

In [ ]:
from sklearn.cluster import KMeans #nhập KMeanslớp từ sklearn.clustermô-đun. Scikit-learn là một thư viện học máy được sử dụng rộng rãi trong Python.
kmeans = KMeans(n_clusters=3, random_state=0).fit(X) #tạo ra một KMeansđối tượng và ngay lập tức khớp nó với dữ liệu của bạn X, chỉ rõ rằng bạn muốn tìm 3 cụm. ặt một hạt giống cho trình tạo số ngẫu nhiên. Điều này đảm bảo kết quả có thể lặp lại mỗi lần bạn chạy mã. .fit(X)đào tạo mô hình K-means trên dữ liệu của bạn X.
print('Centers found by scikit-learn:') #in ra thông báo mô tả vào bảng điều khiển.
print(kmeans.cluster_centers_) #in ra các tâm cụm được KMeansmô hình scikit-learn tìm thấy sau khi khớp. cluster_centers_Thuộc tính của KMeansđối tượng lưu trữ tọa độ của các tâm cụm cuối cùng.
pred_label = kmeans.predict(X) #sử dụng mô hình K-means scikit-learn đã được huấn luyện để dự đoán nhãn cụm cho mỗi điểm dữ liệu trong X. predict()Phương pháp này gán mỗi điểm dữ liệu cho cụm có tâm gần nhất. Các nhãn kết quả được lưu trữ trong pred_labelbiến.
kmeans_display(X, pred_label) #gọi kmeans_displayhàm tùy chỉnh của bạn (được xác định trước đó) để trực quan hóa các điểm dữ liệu Xbằng cách sử dụng nhãn cụm được dự đoán bởi mô hình scikit-learn ( pred_label).
#Tóm lại, đoạn mã này cho thấy cách đạt được kết quả phân cụ K-means tương tự bằng cách sử dụng triển khai tối ưu của scikit-learn và so sánh trực quan kết quả đầu ra của nó (các phép gán cụm) với đoạn mã K-means của riêng bạn.

## Ghi chú học tập / Study prompts

- Đầu vào của từng hàm là gì, đầu ra có kiểu và kích thước nào?
- Thay đổi tham số hoặc dữ liệu nhỏ sẽ ảnh hưởng kết quả ra sao?
- Cell nào cần dữ liệu bên ngoài, thư viện hoặc thao tác nhập tay?

Explain the data flow, modify one example, and record the result when revisiting this exercise.